# MicroDuck PPO with backend-neutral MuJoCo

This notebook trains a deliberately small stand/balance task on a laptop CPU. The task is written once against `MujocoEnv`; changing `BACKEND` to `"mjx"` or `"mujoco-torch"` changes the physics implementation without changing the observation, action, reward, or termination definitions.

From the TorchRL checkout, start Jupyter with:

```bash
uv run --extra notebook --with mujoco --with matplotlib \
  --with 'nodejs-wheel>=22.13,<25' \
  jupyter lab examples/mujoco/microduck_ppo.ipynb
```

The temporary `nodejs-wheel` dependency supplies Node.js and npm for the final MuJoCo WASM viewer cell. You can omit it when `node` and either `npm` or `pnpm` are already on `PATH`.

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torchrl.envs import ExplorationType, set_exploration_type
from torchrl.envs.utils import check_env_specs
from torchrl.render import (
    display_mujoco_wasm_viewer,
    extract_qpos_trajectory,
    play_mujoco_wasm_trajectory,
    write_mujoco_wasm_viewer,
)

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "torchrl").is_dir() and (path / "examples").is_dir()
)
sys.path.insert(0, str(repo_root))

from examples.mujoco.ppo_microduck import (
    MicroDuckStandEnv,
    collect_qpos_trajectory,
    make_env,
    make_models,
    resolve_microduck_scene,
    train_ppo,
)

## Locate the shared MJCF

The XML and meshes stay in `microduck_rl`; this notebook does not fork or translate them. Set `MICRODUCK_RL_ROOT` to an existing checkout. For a first run, the cell below downloads the repository's source archive next to the TorchRL checkout when one is not already present. This avoids requiring Git or GitHub credentials.

In [ ]:
microduck_root = Path(
    os.environ.get("MICRODUCK_RL_ROOT", repo_root.parent / "microduck_rl")
).expanduser()
if not microduck_root.exists():
    microduck_root.parent.mkdir(parents=True, exist_ok=True)
    archive_url = (
        "https://github.com/pollen-robotics/microduck_rl/"
        "archive/refs/heads/main.zip"
    )
    with tempfile.TemporaryDirectory(prefix="microduck_rl_download_") as tmp_dir:
        archive_path = Path(tmp_dir) / "microduck_rl.zip"
        urllib.request.urlretrieve(archive_url, archive_path)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(tmp_dir)
        shutil.move(Path(tmp_dir) / "microduck_rl-main", microduck_root)
scene_path = resolve_microduck_scene(microduck_root)
scene_path

## The first reward

The action is a 14-dimensional normalized joint-position offset around the MJCF's `STAND` keyframe. The 48-dimensional observation is projected gravity (3), base angular velocity (3), joint-position error (14), joint velocity (14), and the previous action (14).

The reward is

$$2 r_{upright} + 0.5 r_{height} + 0.5 r_{pose} - 0.02 c_{angular\ velocity} - 0.002 c_{joint\ velocity} - 0.02 c_{action\ rate}.$$

The three positive terms are smooth exponentials. An episode terminates when the torso gets too low, the duck tilts past the allowed angle, or state becomes non-finite. This is intentionally a legible first task: once it learns, locomotion can add a commanded planar velocity term without changing the backend interface.

In [ ]:
BACKEND = "mujoco"  # "mujoco", "mjx", or "mujoco-torch"
DEVICE = "cpu"
NUM_ENVS = 4

env = make_env(
    scene_path,
    backend=BACKEND,
    num_envs=NUM_ENVS,
    device=DEVICE,
    seed=0,
)
check_env_specs(env)
reset_td = env.reset()
reset_td["observation"].shape, env.action_spec.shape

## Native render before training

Rendering uses the official MuJoCo renderer on CPU. Training keeps pixels out of the hot loop.

In [ ]:
render_env = MicroDuckStandEnv(
    scene_path,
    backend="mujoco",
    num_envs=1,
    reset_noise_scale=0.0,
    render_width=640,
    render_height=480,
)
render_env.reset()
frame = render_env.render()[0].cpu()
plt.figure(figsize=(8, 6))
plt.imshow(frame)
plt.axis("off")
plt.show()

## A short PPO run

Four iterations are only a pipeline check, not a solved policy. Increase `iterations`, `NUM_ENVS`, and `rollout_steps` for a meaningful run. Native MuJoCo uses serial vectorization in this notebook so it remains reliable inside a Jupyter kernel.

In [ ]:
torch.manual_seed(0)
actor, critic = make_models(env, device=DEVICE)
history = train_ppo(
    env,
    actor,
    critic,
    iterations=4,
    rollout_steps=64,
    epochs=3,
    minibatch_size=128,
)
history

In [ ]:
plt.plot([row["iteration"] for row in history], [row["reward"] for row in history])
plt.xlabel("PPO iteration")
plt.ylabel("mean step reward")
plt.show()

## Interactive MuJoCo WASM viewer

`torchrl.render` copies the same XML, included XML files, and meshes into a local browser viewer. The iframe exposes all 14 joint sliders, Reset, Step, and Run physics controls. It also plays the qpos trajectory collected from the policy. Moving a slider interrupts playback, so the duck remains directly controllable. The first execution installs the small JavaScript viewer dependencies locally.

In [ ]:
with set_exploration_type(ExplorationType.DETERMINISTIC):
    rollout = collect_qpos_trajectory(render_env, actor, steps=300)
qpos_trajectory = extract_qpos_trajectory(rollout)
len(qpos_trajectory), len(qpos_trajectory[0])

In [ ]:
viewer_dir = Path(tempfile.mkdtemp(prefix="torchrl_microduck_wasm_"))
write_mujoco_wasm_viewer(viewer_dir, scene_path)
viewer_process = display_mujoco_wasm_viewer(viewer_dir, height=720)
play_mujoco_wasm_trajectory(
    qpos_trajectory,
    fps=30,
    loop=True,
    viewer_dir=viewer_dir,
    viewer_origin=viewer_process.torchrl_mujoco_wasm_origin,
)

## Next experiments

A useful next reward revision is commanded locomotion: add target linear and yaw velocities to the observation, reward velocity tracking, and retain upright/height/action-rate terms as stabilizers. MJLab-specific actuator identification, delays, and domain randomization should remain explicit optional task layers rather than being hidden in the shared MJCF.

In [ ]:
viewer_process.terminate()
env.close()
render_env.close()